In [1]:
import pandas as pd

# Load the single source of truth into memory
manifest_df = pd.read_csv("../data/processed/manifest.csv")

# Take a peek to make sure it loaded correctly
manifest_df.head()

,filepath,filename,split,label_name,extension,file_size_bytes,valid_extension,non_empty,is_readable,width,height,mode,file_hash,expected_mode,is_duplicate,is_valid,label,processed_filepath
0,..\data\raw\test\def_front\cast_def_0_1059.jpeg,cast_def_0_1059.jpeg,test,def_front,.jpeg,10701,True,True,True,300,300,RGB,4e26d6cf05102659baa357b0405d3b97,False,False,True,1,..\data\processed\test\def_front\cast_def_0_10...
1,..\data\raw\test\def_front\cast_def_0_1063.jpeg,cast_def_0_1063.jpeg,test,def_front,.jpeg,9656,True,True,True,300,300,RGB,110f4382c2e4897c5ef7405e15de1946,False,False,True,1,..\data\processed\test\def_front\cast_def_0_10...
2,..\data\raw\test\def_front\cast_def_0_108.jpeg,cast_def_0_108.jpeg,test,def_front,.jpeg,11100,True,True,True,300,300,RGB,d8700301b355952e83eb3620218b0652,False,False,True,1,..\data\processed\test\def_front\cast_def_0_10...
3,..\data\raw\test\def_front\cast_def_0_1096.jpeg,cast_def_0_1096.jpeg,test,def_front,.jpeg,9797,True,True,True,300,300,RGB,695f2a02f07c7f6831f3818bca2e7909,False,False,True,1,..\data\processed\test\def_front\cast_def_0_10...
4,..\data\raw\test\def_front\cast_def_0_112.jpeg,cast_def_0_112.jpeg,test,def_front,.jpeg,11642,True,True,True,300,300,RGB,b22cbc17c6f36f3cb9192ea01e23c08b,False,False,True,1,..\data\processed\test\def_front\cast_def_0_11...


In [2]:
import torch
print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.13.0+cpu


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import random
import mlflow
from pathlib import Path
from sklearn.model_selection import train_test_split

# ==========================================
# A. FIX: SET SEED FOR REPRODUCIBILITY
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ==========================================
# B. FIX: POINT MLFLOW TO THE SHARED DATABASE
# ==========================================
PROJECT_ROOT = Path("..").resolve()
MLFLOW_DB = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")

# 1. Load the Manifest & Split (FIX: Added Validation Split)
manifest_df = pd.read_csv("../data/processed/manifest.csv")
train_full = manifest_df[manifest_df['split'] == 'train']
test_df = manifest_df[manifest_df['split'] == 'test']

train_df, val_df = train_test_split(
    train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=train_full["label"]
)

# 2. Define the Dataset Class
class CastingDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['processed_filepath']
        image = Image.open(img_path).convert('RGB') 
        label = self.dataframe.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

# 3. Apply standard PyTorch Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. Initialize the Loaders (FIX: Added val_loader)
train_dataset = CastingDataset(train_df, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = CastingDataset(val_df, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

test_dataset = CastingDataset(test_df, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 5. TRANSFER LEARNING: Load the Expert Model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 1),
    nn.Sigmoid() 
)

print("Data Loaders and ResNet18 Model ready!")

c:\Users\G689507\ml-engineering-miniproject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Loaders and ResNet18 Model ready!


In [4]:
import torch.optim as optim
import torch.nn as nn
import mlflow

criterion = nn.BCELoss() 
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# FIX: Use the exact same experiment name as the CNN notebook
mlflow.set_experiment("casting-quality-cnn")

with mlflow.start_run(run_name="ResNet18_Base_Run"):
    
    epochs = 3  
    mlflow.log_param("model_architecture", "ResNet18")
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("epochs", epochs)
    
    print(f"Starting Training for {epochs} Epochs on CPU...\n")

    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train() 
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            predictions = (outputs.squeeze() > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = correct / total

        # --- FIX: VALIDATION PHASE ---
        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for val_images, val_labels in val_loader:
                val_outputs = model(val_images)
                v_loss = criterion(val_outputs.squeeze(), val_labels)
                
                val_running_loss += v_loss.item()
                val_predictions = (val_outputs.squeeze() > 0.5).float()
                val_correct += (val_predictions == val_labels).sum().item()
                val_total += val_labels.size(0)
                
        val_loss = val_running_loss / len(val_loader)
        val_accuracy = val_correct / val_total

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_accuracy:.2%} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2%}")
        
        # Log to MLflow
        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        mlflow.log_metric("train_accuracy", epoch_accuracy, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_accuracy", val_accuracy, step=epoch)

    print("\nTraining Complete!")

Starting Training for 3 Epochs on CPU...

Epoch 1/3 | Train Loss: 0.3937 | Train Acc: 84.62% | Val Loss: 0.2318 | Val Acc: 94.29%
Epoch 2/3 | Train Loss: 0.2163 | Train Acc: 94.01% | Val Loss: 0.1599 | Val Acc: 97.41%
Epoch 3/3 | Train Loss: 0.1649 | Train Acc: 95.78% | Val Loss: 0.1337 | Val Acc: 96.88%

Training Complete!


In [5]:
# 1. Put the model in evaluation mode (turns off learning)
model.eval()

correct = 0
total = 0

print("Evaluating model on the unseen test dataset...")

# 2. Turn off the gradient engine (saves memory since we aren't updating weights)
with torch.no_grad():
    for images, labels in test_loader:
        
        # Make guesses on the test images
        outputs = model(images)
        predictions = (outputs.squeeze() > 0.5).float()
        
        # Tally up the correct guesses
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

# 3. Calculate and log the final score
test_accuracy = correct / total
print(f"Final Test Accuracy: {test_accuracy:.2%}")

# Log it to MLflow so Person 2 can see your final score!
with mlflow.start_run(run_name="ResNet18_Base_Run", nested=True):
    mlflow.log_metric("final_test_accuracy", test_accuracy)

Evaluating model on the unseen test dataset...
Final Test Accuracy: 97.48%


In [9]:
model.eval()

actual_defects = 0
defects_caught = 0

print("Calculating defect detection rates...")

with torch.no_grad():
    for images, labels in test_loader:
        
        outputs = model(images)
        predictions = (outputs.squeeze() > 0.5).float()
        
        # 1. Count how many images in this batch are ACTUALLY defective (label == 1)
        actual_defects += (labels == 1).sum().item()
        
        # 2. Count how many the model correctly PREDICTED as defective
        # (Where prediction is 1 AND label is 1)
        defects_caught += ((predictions == 1) & (labels == 1)).sum().item()

# Calculate the final catch rate
if actual_defects > 0:
    catch_rate = defects_caught / actual_defects
else:
    catch_rate = 0.0

print(f"Total Actual Defects in Test Set: {actual_defects}")
print(f"Defects Successfully Caught: {defects_caught}")
print(f"Defect Catch Rate (Recall): {catch_rate:.2%}")

Calculating defect detection rates...
Total Actual Defects in Test Set: 453
Defects Successfully Caught: 435
Defect Catch Rate (Recall): 96.03%


In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

print("Evaluating model and calculating all metrics...")

with torch.no_grad():
    for images, labels in test_loader:
        # Note: model.fc already contains a Sigmoid, so outputs are probabilities
        probabilities = model(images).squeeze()
        predictions = (probabilities > 0.5).float()
        
        all_labels.extend(labels.numpy().astype(int))
        all_predictions.extend(predictions.numpy().astype(int))
        all_probabilities.extend(probabilities.numpy())

# FIX: Calculate all metrics to match the CNN scorecard
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions, zero_division=0)
recall = recall_score(all_labels, all_predictions, zero_division=0)
f1 = f1_score(all_labels, all_predictions, zero_division=0)
roc_auc = roc_auc_score(all_labels, all_probabilities)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC AUC  : {roc_auc:.4f}")

# Log EVERYTHING to MLflow
with mlflow.start_run(run_name="ResNet18_Evaluation", nested=True):
    mlflow.log_metrics({
        "test_accuracy": float(accuracy),
        "test_precision": float(precision),
        "test_recall": float(recall),
        "test_f1": float(f1),
        "test_roc_auc": float(roc_auc)
    })
    
    print("\nSuccessfully logged all metrics to MLflow!")

Evaluating model and calculating all metrics...
Accuracy : 0.9748
Precision: 1.0000
Recall   : 0.9603
F1 Score : 0.9797
ROC AUC  : 0.9988

Successfully logged all metrics to MLflow!


In [8]:
import os
import torch

# 1. Ensure the models directory exists at the root level
models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

# 2. Define the exact file path
save_path = os.path.join(models_dir, "resnet18_transfer_best.pth")

# 3. Save the trained model's state dictionary
torch.save(model.state_dict(), save_path)

print(f"Model successfully saved to: {save_path}")

Model successfully saved to: ../models\resnet18_transfer_best.pth
